In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round as spark_round
)

In [0]:
silver_table = "online_retail.silver.transactions_clean"
gold_table = "online_retail.gold.product_sales_summary"

stored_silver_df = spark.table(silver_table)
stored_gold_df = spark.table(gold_table)

In [0]:
# Calculate totals from qualifying Silver sales

silver_summary_df = (
    stored_silver_df
    .filter(col("is_positive_sale"))
    .agg(
        spark_sum(
            col("quantity")
        ).alias("total_quantity"),

        spark_round(
            spark_sum(col("line_total")),
            2
        ).alias("total_revenue")
    )
)

silver_summary = silver_summary_df.first()

In [0]:
# Calculate totals from the Gold summary

gold_summary_df = (
    stored_gold_df
    .agg(
        spark_sum(
            col("total_quantity_sold")
        ).alias("total_quantity"),

        spark_round(
            spark_sum(col("total_revenue")),
            2
        ).alias("total_revenue")
    )
)

gold_summary = gold_summary_df.first()

In [0]:
# Compare Silver and Gold totals
silver_total_quantity = silver_summary["total_quantity"]
gold_total_quantity = gold_summary["total_quantity"]

silver_total_revenue = silver_summary["total_revenue"]
gold_total_revenue = gold_summary["total_revenue"]

quantity_matches = (silver_total_quantity == gold_total_quantity)
revenue_matches = (silver_total_revenue == gold_total_revenue)

pipeline_validation_passed = (quantity_matches and revenue_matches)

print(f"Silver total quantity: {silver_total_quantity:,}")
print(f"Gold total quantity: {gold_total_quantity:,}")
print(f"Quantities match: {quantity_matches}")

print(f"Silver total revenue: {silver_total_revenue:,.2f}")
print(f"Gold total revenue: {gold_total_revenue:,.2f}")
print(f"Revenues match: {revenue_matches}")

print(f"Pipeline validation passed: {pipeline_validation_passed}")

Silver total quantity: 425,461
Gold total quantity: 425,461
Quantities match: True
Silver total revenue: 822,483.95
Gold total revenue: 822,483.95
Revenues match: True
Pipeline validation passed: True
